# 잘못된 주문을 걸러 봐요

`check_order`는 주문을 처리해도 되는지 확인하는 함수예요. 비어 있는 조건 두 곳을 바꾸고, 결과를 파일로 저장해요.

- 처음에는 통과 10건·거부 1건·놓침 1건이 나와요. 아직 조건이 완성되지 않았기 때문이에요.
- 두 조건을 채우면 통과 10건·거부 2건·놓침 0건, 합계 112,250원이 나와요.
- 수량 `0`과 `"두"`도 거부되는지 확인해요. 기본 12건만으로는 수량 조건까지 확인할 수 없어요.

## 시작하기

1. 위 메뉴에서 **파일 → Drive에 사본 저장**을 눌러요. 내 사본에 답을 남길 수 있어요.
2. 첫 코드 칸 왼쪽의 **▶**를 눌러요. `준비 완료`가 나오면 다음 칸으로 가요.
3. 수업 중에는 안내한 칸만 실행해요. **Shift+Enter**도 같은 칸을 실행하는 방법이에요.

`Cell`은 코드나 설명이 들어 있는 칸이에요. 런타임이 초기화되어 파일이나 변수가 사라졌다면 첫 칸부터 다시 실행해요. 단순히 브라우저를 다시 여는 것과는 달라요.

코드 앞의 `#`는 설명이에요. 실행되지 않아요. `import`는 다른 파일의 이름을 가져오고, 줄 앞의 `!`는 Python 대신 터미널 명령을 실행해요.


In [ ]:
# ── 첫 Cell · 준비 ──────────────────────────────────────────
# Colab은 구글이 빌려주는 컴퓨터입니다. 새로 켤 때마다 빈 컴퓨터이므로
# 필요한 파일을 GitHub에서 받아 와야 합니다. 이 Cell이 그 일을 합니다.
#
# 실행하는 법: 이 Cell을 클릭한 뒤 왼쪽 ▶ 버튼을 누르거나 Shift+Enter.
# 위 메뉴 [런타임 → 모두 실행]을 누르면 위에서 아래로 전부 실행됩니다.

import os, sys          # import = 파이썬에 이미 들어 있는 도구 상자를 꺼내는 일

# 줄 앞의 ! 는 "파이썬이 아니라 터미널 명령"이라는 표시입니다.
# git clone = GitHub에 있는 폴더를 통째로 이 컴퓨터로 복사하는 명령.
if not os.path.isdir("jnu-llmops-precourse-day2"):      # 이미 받았으면 건너뜁니다
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 파이썬이 import 할 파일을 찾는 폴더 목록에 어제의 solution 폴더를 넣습니다.
# 이 줄이 없으면 아래 Cell의 from order import Order 가 파일을 못 찾습니다.
sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")

## 1. 주문 기록을 읽어요

다음 칸을 실행하면 기록 수 `12`와 첫 메뉴 `카페라떼`가 나와요. `FileNotFoundError`가 나오면 첫 준비 칸의 실행 결과부터 확인하세요.

In [ ]:
import json                            # JSON 글자를 파이썬 값으로 바꾸는 도구
from catalog import MENU
from order import Order
from pricing import calculate_bill

# open = 파일을 연다, json.load = 그 글자를 읽어 파이썬 값(목록·딕셔너리)으로 바꾼다
with open("jnu-llmops-precourse-day3/data/orders.json", encoding="utf-8") as f:
    records = json.load(f)

print(len(records))                          # 기록이 몇 건인가
print(records[0]["items"][0]["menu_name"])   # 첫 기록 → items 목록 → 첫 항목 → 이름

## 2. 검사 없이 처리하면 어디서 멈출까요?

다음 칸을 실행하면 A01부터 A06까지 계산한 뒤 `KeyError`가 나와요. A07에 있는 `녹차라떼`가 메뉴판에 없기 때문이에요.

이 오류는 실습에서 확인할 결과예요. 노트북이 고장 난 것은 아니에요. 오류를 출력하도록 해 두어서 다음 칸도 실행할 수 있어요.

In [ ]:
import traceback
try:
    for record in records:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        print(record["order_id"], bill.total)
except Exception:
    traceback.print_exc()

## 3. False 두 곳을 조건으로 바꿔요

1번 필수 이름 검사는 완성돼 있어요. 그 모양을 보고 다음 두 조건을 채워 보세요.

- 2번: 메뉴판 `MENU`에 없는 메뉴인가요?
- 3번: 수량이 정수가 아니거나, 1~10 밖인가요?

함수를 바꾼 뒤에는 **그 코드 칸부터 다시 실행**해야 변경이 적용돼요. 그다음 짧은 확인 칸을 실행해 A07·A11의 거부 이유를 읽어요.

`True`는 통과, `False`는 거부예요. 이 함수는 오늘 데이터에 필요한 검사만 하며 모든 형태의 주문을 검사하지는 않아요.

In [ ]:
REQUIRED_KEYS = ("order_id", "items", "is_student")

def check_order(record):
    for key in REQUIRED_KEYS:                       # 1. 필수 Key  (완성 예)
        if key not in record:
            return False, f"필수 Key 없음: {key}"
    for item in record["items"]:
        if False:                                   # 2. 허용 메뉴  ← 이 조건을 채웁니다
            return False, f"없는 메뉴: {item['menu_name']}"
        quantity = item["quantity"]
        if False:                                   # 3. 수량 범위  ← 이 조건을 채웁니다
            return False, f"수량 범위 밖: {quantity!r}"
    return True, ""


In [ ]:
print(check_order(records[0]))    # 기대 (True, '')
print(check_order(records[6]))    # 기대 (False, '없는 메뉴: 녹차라떼')
print(check_order(records[10]))   # 기대 (False, '필수 Key 없음: is_student')

## 4. 건수와 거부 이유를 비교해요

다음 두 칸을 실행해 관찰값과 기대값을 비교해요.

- 통과 10건, 거부 2건, 놓침 0건
- 통과 합계 112,250원
- A07: 없는 메뉴, A11: 필수 이름 누락

`놓침`은 검사를 통과했지만 계산하다 실패한 주문이에요. 놓침이 남으면 3번의 조건을 확인하고 다시 실행하세요.

In [ ]:
accepted, rejected, missed = [], [], []
for record in records:
    ok, reason = check_order(record)
    if not ok:
        rejected.append({"order_id": record["order_id"], "reason": reason})
        continue
    try:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        accepted.append({"order_id": record["order_id"], "total": bill.total})
    except Exception as e:                     # 문지기가 놓친 것은 여기서 드러납니다
        missed.append({"order_id": record["order_id"], "error": f"{type(e).__name__}: {e}"})

print("통과", len(accepted), "건 · 거부", len(rejected), "건 · 놓침", len(missed), "건 · 통과 합계", sum(a["total"] for a in accepted), "원")
for r in rejected:
    print("거부", r["order_id"], "-", r["reason"])
for m in missed:
    print("놓침", m["order_id"], "-", m["error"], "← 문지기가 먼저 거부했어야 합니다")

In [ ]:
with open("jnu-llmops-precourse-day3/data/expected_day3.json", encoding="utf-8") as f:
    expected = json.load(f)
print("기대:", expected["accepted"], "건 통과 ·", expected["rejected"], "건 거부 · 합계", expected["accepted_total"], "원")
print("관찰:", len(accepted), "건 통과 ·", len(rejected), "건 거부 · 합계", sum(a["total"] for a in accepted), "원")

## 5. 결과 파일을 저장해요

다음 칸을 실행하면 `orders_result.json`이 생겨요. 왼쪽 파일 목록에서 파일 오른쪽 메뉴를 누르고 **다운로드**를 선택하세요.

이 파일에는 통과·거부 목록만 들어가요. `놓침`은 저장되지 않으므로, 놓침이 남아 있다면 완성 결과로 제출하지 않아요. 수량 검사는 아래에서 한 번 더 확인해요.

In [ ]:
with open("orders_result.json", "w", encoding="utf-8") as f:
    json.dump({"accepted": accepted, "rejected": rejected}, f, ensure_ascii=False, indent=2)
print("저장:", "orders_result.json")

## 6. 수량 조건도 확인해요

다음 칸을 실행하면 수량 `0`인 X01과 수량 `"두"`인 X02의 검사 결과가 나와요. **둘 다 False와 거부 이유가 나오는지** 확인해요.

True가 나오면 수량 조건을 다시 보세요. 조건 칸부터 결과 저장 칸까지 다시 실행하고, 이 칸도 다시 확인해요. 두 검사는 모든 참가자가 확인해요.

In [ ]:
extra = [
    {"order_id": "X01", "items": [{"menu_name": "카페라떼", "quantity": 0}], "is_student": False},
    {"order_id": "X02", "items": [{"menu_name": "카페라떼", "quantity": "두"}], "is_student": False},
]
for record in extra:
    print(record["order_id"], check_order(record))

## 확인한 내용을 세 문장으로 남겨요

기대값을 옮기기보다 내 화면에서 본 내용을 적어요.

- 입력: 어떤 파일에서 몇 건을 읽었나요?
- 결과: 통과·거부·놓침은 몇 건이고, 수량 검사 두 건은 어떻게 나왔나요?
- 변경: 어떤 조건을 바꿨고, 아직 확인하지 못한 것은 무엇인가요?